# Ejecucion de random forest

In [34]:
%pip install pandas numpy scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


## Generación del modelo

In [36]:
def generate_model_rf(X_train, X_val, y_train, y_val, n_estimators, max_depth, min_samples_split):
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    r2 = r2_score(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)

    return r2, mae, mse, rmse

## Entrenamiento principal para entrenar y comparar combinaciones de hiperparámetros

In [37]:
def train_model_with_hyperparameters():
    # Cargar archivos
    df_train = pd.read_csv('../Data/train.csv')
    df_val = pd.read_csv('../Data/validation.csv')
    df_test = pd.read_csv('../Data/test.csv')

    # Combinar train + validation para codificar categóricas de forma uniforme
    df_combined = pd.concat([df_train, df_val])
    label_encoders = {}
    for col in df_combined.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df_combined[col] = le.fit_transform(df_combined[col].astype(str))
        label_encoders[col] = le

    # Repartir codificados
    df_train = df_combined.iloc[:len(df_train)].copy()
    df_val = df_combined.iloc[len(df_train):].copy()

    # Aplicar la misma codificación a test
    for col in label_encoders:
        df_test[col] = label_encoders[col].transform(df_test[col].astype(str))

    # Definir X e y
    X_train = df_train.drop(columns=['G1', 'G2', 'G3'])
    y_train = df_train['G3']
    X_val = df_val.drop(columns=['G1', 'G2', 'G3'])
    y_val = df_val['G3']

    # Combinaciones de hiperparámetros
    n_estimators_list = [100, 350]
    max_depth_list = [10, 40]
    min_samples_split_list = [12, 4]

    # Evaluación de combinaciones
    for n in n_estimators_list:
        for d in max_depth_list:
            for s in min_samples_split_list:
                r2, mae, mse, rmse = generate_model_rf(X_train, X_val, y_train, y_val, n, d, s)
                print(f"Combinación: n_estimators={n}, max_depth={d}, min_samples_split={s}")
                print(f"R²: {r2:.4f} | MAE: {mae:.4f} | MSE: {mse:.4f} | RMSE: {rmse:.4f}")
                print('-' * 70)


In [38]:
train_model_with_hyperparameters()

Combinación: n_estimators=100, max_depth=10, min_samples_split=12
R²: 0.2891 | MAE: 3.2352 | MSE: 16.0340 | RMSE: 4.0043
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=10, min_samples_split=4
R²: 0.3001 | MAE: 3.1946 | MSE: 15.7868 | RMSE: 3.9733
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=40, min_samples_split=12
R²: 0.2892 | MAE: 3.2364 | MSE: 16.0305 | RMSE: 4.0038
----------------------------------------------------------------------
Combinación: n_estimators=100, max_depth=40, min_samples_split=4
R²: 0.3025 | MAE: 3.1983 | MSE: 15.7312 | RMSE: 3.9663
----------------------------------------------------------------------
Combinación: n_estimators=350, max_depth=10, min_samples_split=12
R²: 0.2982 | MAE: 3.2341 | MSE: 15.8282 | RMSE: 3.9785
----------------------------------------------------------------------
Combinación: n_estimators=350, max_depth=1

## Análisis

Los resultados de **Random Forest** muestran una ligera mejora respecto a los modelos **Ridge** y **Lasso**, con valores de **R²** que oscilan entre **0.30** y **0.32**, indicando que el modelo captura solo una pequeña parte de la variabilidad de las notas finales. A pesar de esta mejora, los errores como **MAE** y **MSE** siguen siendo moderados, con predicciones que se desvían entre **3.2** y **3.9** puntos de la realidad. El aumento del número de estimadores y la profundidad del árbol mejora ligeramente el ajuste, pero los resultados siguen siendo limitados, lo que sugiere que se requieren ajustes adicionales en los hiperparámetros o modelos más complejos para obtener un mejor rendimiento.